In [1]:

# Step 1: Check GPU and mount Google Drive

import torch
import os
import pandas as pd
from google.colab import drive

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected. Enable a GPU runtime first.")

drive.mount('/content/drive')

# Check research folder
DATA_DIR = "/content/drive/MyDrive/Task1_Research"

print("\nResearch folder exists:", os.path.exists(DATA_DIR))

if os.path.exists(DATA_DIR):
    print("\nFiles in research folder:")
    for filename in sorted(os.listdir(DATA_DIR)):
        print(filename)


PyTorch version: 2.11.0+cu130
GPU available: True
GPU name: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Research folder exists: True

Files in research folder:
distilled_student_model
distilled_student_training_history.csv
finetuned_teacher_7b
finetuned_teacher_kd_student
finetuned_teacher_kd_student_test_metrics.csv
finetuned_teacher_kd_student_test_predictions.csv
finetuned_teacher_soft_targets_train.csv
finetuned_teacher_validation_audit.csv
native_student_model
native_student_rawtext_controlled
native_student_test_metrics.csv
native_student_training_history.csv
native_student_validation_audit.csv
original_kd_audit
original_kd_student_test_metrics.csv
task1_baseline_metrics.csv
task1_data_leakage_audit.csv
task1_label_audit_sample.csv
task1_labeling_rules.txt
task1_manual_review_candidates.csv
task1_teacher_soft_targets_checkpoint.csv
task1_teacher_soft_targets_pilot.csv
task1_teacher_soft

In [1]:

import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("GPU is still unavailable. Check runtime settings.")


PyTorch version: 2.11.0+cu130
GPU available: True
GPU name: Tesla T4
CUDA version: 13.0


In [4]:


import os

from google.colab import drive
drive.mount('/content/drive')

root = "/content/drive/MyDrive"

print("MyDrive exists:", os.path.exists(root))

# Search for the three dataset files
targets = {
    "task1_train.csv",
    "task1_validation.csv",
    "task1_test.csv"
}

found = {}

for current_root, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if d not in {
        ".git", "node_modules", ".cache"
    }]

    for filename in files:
        if filename in targets:
            path = os.path.join(current_root, filename)
            found[filename] = path
            print(filename, "->", path)

print("\nFiles found:", len(found), "of", len(targets))


Mounted at /content/drive
MyDrive exists: True
task1_validation.csv -> /content/drive/MyDrive/Task1_Research/task1_validation.csv
task1_train.csv -> /content/drive/MyDrive/Task1_Research/task1_train.csv
task1_test.csv -> /content/drive/MyDrive/Task1_Research/task1_test.csv

Files found: 3 of 3


In [5]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

train_df = pd.read_csv(os.path.join(DATA_DIR, "task1_train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "task1_validation.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "task1_test.csv"))

datasets = {
    "TRAIN": train_df,
    "VALIDATION": val_df,
    "TEST": test_df
}

for name, df in datasets.items():
    print(f"\n{'=' * 45}")
    print(name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nLabel distribution:")
    print(df["label"].value_counts(dropna=False))
    print("\nMissing values:")
    print(df[["text", "label"]].isna().sum())
    print("\nExample queries:")
    print(df["text"].head(3).to_list())

# Validate columns, missing values, and labels
expected_labels = {"DATABASE", "GENERAL"}

for name, df in datasets.items():
    assert {"text", "label"}.issubset(df.columns), (
        f"{name}: required columns missing"
    )
    assert df["text"].notna().all(), f"{name}: missing text"
    assert df["label"].notna().all(), f"{name}: missing labels"

    actual_labels = set(df["label"].astype(str).str.upper().unique())
    assert actual_labels == expected_labels, (
        f"{name}: unexpected labels {actual_labels}"
    )

print("\n✅ Dataset validation passed.")



TRAIN
Shape: (4000, 2)
Columns: ['text', 'label']

Label distribution:
label
DATABASE    2000
GENERAL     2000
Name: count, dtype: int64

Missing values:
text     0
label    0
dtype: int64

Example queries:
['how many coupless totalled 14.5', 'what are the atmospheric conditions in sparks', 'Can you tell me the highest Total votes that has the % of popular vote of 0.86%, and the # of seats won larger than 0?']

VALIDATION
Shape: (799, 2)
Columns: ['text', 'label']

Label distribution:
label
DATABASE    400
GENERAL     399
Name: count, dtype: int64

Missing values:
text     0
label    0
dtype: int64

Example queries:
['what does idiosyncratic mean', 'what is the weather in tokyo', 'What is the highest value for Byes, when Wins is less than 16, when Benalla DFL is "Swanpool", and when Against is less than 2177?']

TEST
Shape: (1200, 2)
Columns: ['text', 'label']

Label distribution:
label
GENERAL     600
DATABASE    600
Name: count, dtype: int64

Missing values:
text     0
label    0
dt

In [6]:

import os
import torch
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

OUTPUT_DIR = os.path.join(
    DATA_DIR,
    "native_student_rawtext_controlled"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Confirm GPU
assert torch.cuda.is_available(), "GPU is not available!"

print("GPU:", torch.cuda.get_device_name(0))
print("Output folder:", OUTPUT_DIR)
print("Folder exists:", os.path.isdir(OUTPUT_DIR))

# Confirm labels are consistent
for name, df in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)
]:
    df["label"] = df["label"].astype(str).str.upper().str.strip()
    print(f"{name}: {len(df)} rows; labels = {sorted(df['label'].unique())}")

print("\n✅ Experiment folder prepared.")


GPU: Tesla T4
Output folder: /content/drive/MyDrive/Task1_Research/native_student_rawtext_controlled
Folder exists: True
TRAIN: 4000 rows; labels = ['DATABASE', 'GENERAL']
VALIDATION: 799 rows; labels = ['DATABASE', 'GENERAL']
TEST: 1200 rows; labels = ['DATABASE', 'GENERAL']

✅ Experiment folder prepared.


In [7]:

%pip install -q transformers peft accelerate bitsandbytes scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 37.3 MB/s eta 0:00:00


In [8]:

import torch
import transformers
import peft
import accelerate
import bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("Bitsandbytes:", bnb.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU available:", torch.cuda.is_available())

print("\n✅ Library verification complete.")


PyTorch: 2.11.0+cu130
Transformers: 5.18.0
PEFT: 0.21.1
Accelerate: 1.15.0
Bitsandbytes: 0.50.2
GPU: Tesla T4
GPU available: True

✅ Library verification complete.


In [9]:

import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Use 4-bit quantization for the Tesla T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading quantized base model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

# Prepare for 4-bit LoRA training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nGPU:", torch.cuda.get_device_name(0))
print("Model loaded and LoRA configured.")


Loading tokenizer...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826

GPU: Tesla T4
Model loaded and LoRA configured.


In [10]:

# Step 7: Inspect class-token IDs and prepare raw-text examples

import pandas as pd

CLASS_NAMES = ["DATABASE", "GENERAL"]

class_token_ids = {}

for label in CLASS_NAMES:
    token_ids = tokenizer.encode(label, add_special_tokens=False)
    print(f"{label}: token IDs = {token_ids}")

    if len(token_ids) != 1:
        raise ValueError(
            f"{label} is not a single token. "
            "We must adapt the classification objective."
        )

    class_token_ids[label] = token_ids[0]

print("\nClass token IDs:", class_token_ids)

# Check raw query examples and labels
for name, df in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)
]:
    print(f"\n{name} sample:")
    for _, row in df.head(3).iterrows():
        print("Query:", repr(row["text"]))
        print("Label:", row["label"])
        print("---")

print("\n✅ Raw-text data inspection complete.")


DATABASE: token IDs = [46499]
GENERAL: token IDs = [94736]

Class token IDs: {'DATABASE': 46499, 'GENERAL': 94736}

TRAIN sample:
Query: 'how many coupless totalled 14.5'
Label: DATABASE
---
Query: 'what are the atmospheric conditions in sparks'
Label: GENERAL
---
Query: 'Can you tell me the highest Total votes that has the % of popular vote of 0.86%, and the # of seats won larger than 0?'
Label: DATABASE
---

VALIDATION sample:
Query: 'what does idiosyncratic mean'
Label: GENERAL
---
Query: 'what is the weather in tokyo'
Label: GENERAL
---
Query: 'What is the highest value for Byes, when Wins is less than 16, when Benalla DFL is "Swanpool", and when Against is less than 2177?'
Label: DATABASE
---

TEST sample:
Query: 'how many calories does an orange have'
Label: GENERAL
---
Query: "let's hear a joke!"
Label: GENERAL
---
Query: 'What is the sum of the pick numbers where the player was picked in round 5 and overall pick number was greater than 159?'
Label: DATABASE
---

✅ Raw-text data

In [11]:

import os
import math
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm

# Experiment configuration
MODEL_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(MODEL_DIR, exist_ok=True)

MAX_LENGTH = 256
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4

DATABASE_ID = class_token_ids["DATABASE"]
GENERAL_ID = class_token_ids["GENERAL"]

device = torch.device("cuda")

# Raw text only: no instruction prompt or chat template
class QueryDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["text"].astype(str).tolist()
        self.labels = dataframe["label"].astype(str).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        label = self.labels[idx]
        return {
            "text": self.texts[idx],
            "label_id": 0 if label == "DATABASE" else 1
        }

def collate_fn(batch):
    encoded = tokenizer(
        [item["text"] for item in batch],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt"
    )
    encoded["labels_class"] = torch.tensor(
        [item["label_id"] for item in batch],
        dtype=torch.long
    )
    return encoded

train_loader = DataLoader(
    QueryDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    QueryDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE
)

total_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

def class_logits_from_batch(batch):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False
    )

    # Find the final non-padding token for each example
    last_positions = attention_mask.sum(dim=1) - 1
    rows = torch.arange(input_ids.size(0), device=device)

    final_logits = outputs.logits[rows, last_positions, :]
    return final_logits[:, [DATABASE_ID, GENERAL_ID]]

def evaluate(loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    count = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            labels = batch["labels_class"].to(device)
            logits = class_logits_from_batch(batch)
            loss = torch.nn.functional.cross_entropy(logits, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size

            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            count += batch_size

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return {
        "loss": total_loss / count,
        "accuracy": correct / count,
        "predictions": all_preds,
        "labels": all_labels
    }

history = []
best_val_accuracy = -1.0

optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    seen = 0
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for step, batch in enumerate(progress):
        labels = batch["labels_class"].to(device)
        logits = class_logits_from_batch(batch)
        loss = torch.nn.functional.cross_entropy(logits, labels)
        (loss / GRAD_ACCUM).backward()

        running_loss += loss.item() * labels.size(0)
        seen += labels.size(0)

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        progress.set_postfix(loss=f"{loss.item():.4f}")

    val_metrics = evaluate(val_loader)

    record = {
        "epoch": epoch + 1,
        "train_loss": running_loss / seen,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"]
    }
    history.append(record)

    print(record)

    # Save the best validation checkpoint separately
    if val_metrics["accuracy"] > best_val_accuracy:
        best_val_accuracy = val_metrics["accuracy"]
        model.save_pretrained(MODEL_DIR)
        tokenizer.save_pretrained(MODEL_DIR)
        print("Saved best checkpoint:", MODEL_DIR)

    pd.DataFrame(history).to_csv(
        os.path.join(OUTPUT_DIR, "training_history.csv"),
        index=False
    )

print("\nTraining finished.")
print("Best validation accuracy:", best_val_accuracy)
print("History saved to:", os.path.join(OUTPUT_DIR, "training_history.csv"))


Epoch 1/3:   0%|          | 0/2000 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'epoch': 1, 'train_loss': 0.1219579219875489, 'val_loss': 0.003861601785768447, 'val_accuracy': 0.9987484355444305}
Saved best checkpoint: /content/drive/MyDrive/Task1_Research/native_student_rawtext_controlled/checkpoints


Epoch 2/3:   0%|          | 0/2000 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 1.354336723720273e-05, 'val_loss': 0.0033371614597635297, 'val_accuracy': 0.9987484355444305}


Epoch 3/3:   0%|          | 0/2000 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 1.0786033560279407e-06, 'val_loss': 0.0034217210158084435, 'val_accuracy': 0.9987484355444305}

Training finished.
Best validation accuracy: 0.9987484355444305
History saved to: /content/drive/MyDrive/Task1_Research/native_student_rawtext_controlled/training_history.csv


In [12]:

import os
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from torch.utils.data import DataLoader

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")

# Reload the best saved adapter
eval_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

eval_model = PeftModel.from_pretrained(
    base_model,
    CHECKPOINT_DIR
)
eval_model.eval()
eval_model.config.use_cache = False

db_id = eval_tokenizer.encode(
    "DATABASE", add_special_tokens=False
)
gen_id = eval_tokenizer.encode(
    "GENERAL", add_special_tokens=False
)

assert len(db_id) == 1 and len(gen_id) == 1, (
    "Class labels must each be a single token."
)

class_ids = [db_id[0], gen_id[0]]

test_loader = DataLoader(
    QueryDataset(test_df),
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn
)

all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to("cuda")
        attention_mask = batch["attention_mask"].to("cuda")

        outputs = eval_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False
        )

        last_positions = attention_mask.sum(dim=1) - 1
        rows = torch.arange(input_ids.size(0), device="cuda")
        final_logits = outputs.logits[rows, last_positions, :]

        class_logits = final_logits[:, class_ids]
        predictions = class_logits.argmax(dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(batch["labels_class"].tolist())

label_names = ["DATABASE", "GENERAL"]
accuracy = accuracy_score(all_labels, all_predictions)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    labels=[0, 1],
    average="macro",
    zero_division=0
)

cm = confusion_matrix(all_labels, all_predictions, labels=[0, 1])

print(f"Test accuracy: {accuracy:.4%}")
print(f"Macro precision: {precision:.4f}")
print(f"Macro recall: {recall:.4f}")
print(f"Macro F1: {f1:.4f}")

print("\nClassification report:")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1],
    target_names=label_names,
    digits=4,
    zero_division=0
))

print("Confusion matrix (actual rows, predicted columns):")
print(pd.DataFrame(
    cm,
    index=[f"Actual {x}" for x in label_names],
    columns=[f"Predicted {x}" for x in label_names]
))

# Save predictions and metrics in the new folder
predictions_df = test_df.copy()
predictions_df["actual_label"] = [
    label_names[x] for x in all_labels
]
predictions_df["predicted_label"] = [
    label_names[x] for x in all_predictions
]
predictions_df["correct"] = (
    predictions_df["actual_label"]
    == predictions_df["predicted_label"]
)

predictions_df.to_csv(
    os.path.join(OUTPUT_DIR, "test_predictions.csv"),
    index=False
)

metrics_df = pd.DataFrame([{
    "model": "Native SLM - raw text",
    "test_accuracy": accuracy,
    "macro_precision": precision,
    "macro_recall": recall,
    "macro_f1": f1
}])

metrics_df.to_csv(
    os.path.join(OUTPUT_DIR, "test_metrics.csv"),
    index=False
)

print("\nSaved test predictions and metrics to:", OUTPUT_DIR)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Test accuracy: 99.6667%
Macro precision: 0.9967
Macro recall: 0.9967
Macro F1: 0.9967

Classification report:
              precision    recall  f1-score   support

    DATABASE     0.9950    0.9983    0.9967       600
     GENERAL     0.9983    0.9950    0.9967       600

    accuracy                         0.9967      1200
   macro avg     0.9967    0.9967    0.9967      1200
weighted avg     0.9967    0.9967    0.9967      1200

Confusion matrix (actual rows, predicted columns):
                 Predicted DATABASE  Predicted GENERAL
Actual DATABASE                 599                  1
Actual GENERAL                    3                597

Saved test predictions and metrics to: /content/drive/MyDrive/Task1_Research/native_student_rawtext_controlled


In [13]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"
NEW_DIR = os.path.join(
    DATA_DIR, "native_student_rawtext_controlled"
)

files_to_check = {
    "New Native SLM": os.path.join(
        NEW_DIR, "test_predictions.csv"
    ),
    "Fine-tuned teacher KD": os.path.join(
        DATA_DIR, "finetuned_teacher_kd_student_test_predictions.csv"
    )
}

for name, path in files_to_check.items():
    print(f"\n{'=' * 55}")
    print(name)
    print("Path exists:", os.path.exists(path))

    if not os.path.exists(path):
        continue

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    if {"actual_label", "predicted_label"}.issubset(df.columns):
        print("\nActual labels:")
        print(df["actual_label"].value_counts(dropna=False))
        print("\nPredicted labels:")
        print(df["predicted_label"].value_counts(dropna=False))

        correct = (
            df["actual_label"].astype(str).str.upper()
            == df["predicted_label"].astype(str).str.upper()
        )
        print("\nCorrect:", int(correct.sum()))
        print("Incorrect:", int((~correct).sum()))
        print("Recomputed accuracy:", round(correct.mean(), 6))

        print("\nIncorrect examples (up to 10):")
        print(
            df.loc[~correct].head(10).to_string(index=False)
        )
    else:
        print("Prediction columns not found; inspect this file's schema.")



New Native SLM
Path exists: True
Rows: 1200
Columns: ['text', 'label', 'actual_label', 'predicted_label', 'correct']

Actual labels:
actual_label
GENERAL     600
DATABASE    600
Name: count, dtype: int64

Predicted labels:
predicted_label
DATABASE    602
GENERAL     598
Name: count, dtype: int64

Correct: 1196
Incorrect: 4
Recomputed accuracy: 0.996667

Incorrect examples (up to 10):
                                                         text    label actual_label predicted_label  correct
                          what is the spanish word for garden DATABASE     DATABASE         GENERAL    False
                what kind of calories do chicken tenders have  GENERAL      GENERAL        DATABASE    False
i need interesting baseball trivia from the nineteen eighties  GENERAL      GENERAL        DATABASE    False
          where is the nearest arco gas station to macdonalds  GENERAL      GENERAL        DATABASE    False

Fine-tuned teacher KD
Path exists: True
Rows: 1200
Columns: ['text

In [14]:

import os
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

test_df = pd.read_csv(os.path.join(DATA_DIR, "task1_test.csv"))

kd_path = os.path.join(
    DATA_DIR,
    "finetuned_teacher_kd_student_test_predictions.csv"
)
kd_df = pd.read_csv(kd_path)

print("KD prediction rows:", len(kd_df))
print("Test dataset rows:", len(test_df))
print("KD columns:", kd_df.columns.tolist())

# Confirm expected columns
required = {"text", "label", "predicted_label"}
assert required.issubset(kd_df.columns), (
    f"Missing columns: {required - set(kd_df.columns)}"
)
assert len(kd_df) == len(test_df), "Row counts do not match!"

# Check exact query alignment
same_order = (
    kd_df["text"].astype(str).reset_index(drop=True)
    .equals(test_df["text"].astype(str).reset_index(drop=True))
)

print("Query order matches test set:", same_order)

if same_order:
    # Use the independently loaded test labels as ground truth
    actual = test_df["label"].astype(str).str.upper().str.strip()
    saved_labels = kd_df["label"].astype(str).str.upper().str.strip()

    print(
        "Saved labels match test labels:",
        saved_labels.equals(actual.reset_index(drop=True))
    )
    predicted = (
        kd_df["predicted_label"].astype(str).str.upper().str.strip()
    )
else:
    # Match queries by text instead of assuming row order
    assert not test_df["text"].astype(str).duplicated().any(), (
        "Duplicate test queries found; use a row-level audit."
    )
    assert not kd_df["text"].astype(str).duplicated().any(), (
        "Duplicate KD queries found; use a row-level audit."
    )

    merged = test_df[["text", "label"]].merge(
        kd_df[["text", "predicted_label"]],
        on="text",
        how="outer",
        indicator=True,
        validate="one_to_one"
    )

    print("Text match counts:")
    print(merged["_merge"].value_counts())

    assert (merged["_merge"] == "both").all(), (
        "Some queries did not match. Stop and inspect the unmatched rows."
    )

    actual = merged["label"].astype(str).str.upper().str.strip()
    predicted = (
        merged["predicted_label"].astype(str).str.upper().str.strip()
    )

# Validate prediction labels
valid_labels = {"DATABASE", "GENERAL"}
assert set(predicted.unique()).issubset(valid_labels), (
    f"Unexpected predicted labels: {set(predicted.unique())}"
)

accuracy = accuracy_score(actual, predicted)

print(f"\nRecomputed test accuracy: {accuracy:.4%}")
print("\nClassification report:")
print(classification_report(
    actual,
    predicted,
    labels=["DATABASE", "GENERAL"],
    digits=4,
    zero_division=0
))

print("Confusion matrix (actual rows, predicted columns):")
print(pd.DataFrame(
    confusion_matrix(
        actual,
        predicted,
        labels=["DATABASE", "GENERAL"]
    ),
    index=["Actual DATABASE", "Actual GENERAL"],
    columns=["Predicted DATABASE", "Predicted GENERAL"]
))


KD prediction rows: 1200
Test dataset rows: 1200
KD columns: ['text', 'label', 'predicted_label', 'confidence']
Query order matches test set: True
Saved labels match test labels: True

Recomputed test accuracy: 99.9167%

Classification report:
              precision    recall  f1-score   support

    DATABASE     1.0000    0.9983    0.9992       600
     GENERAL     0.9983    1.0000    0.9992       600

    accuracy                         0.9992      1200
   macro avg     0.9992    0.9992    0.9992      1200
weighted avg     0.9992    0.9992    0.9992      1200

Confusion matrix (actual rows, predicted columns):
                 Predicted DATABASE  Predicted GENERAL
Actual DATABASE                 599                  1
Actual GENERAL                    0                600


In [15]:

import os
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "task1_test.csv")
)

original_metrics_path = os.path.join(
    DATA_DIR, "original_kd_student_test_metrics.csv"
)

print("Original KD metrics file exists:",
      os.path.exists(original_metrics_path))

if os.path.exists(original_metrics_path):
    print(pd.read_csv(original_metrics_path).to_string(index=False))

# Find possible saved prediction files in the research folder
print("\nPossible original KD prediction files:")
for filename in os.listdir(DATA_DIR):
    if "original_kd" in filename.lower() and "prediction" in filename.lower():
        print(filename)


Original KD metrics file exists: True
          model  accuracy  macro_precision  macro_recall  macro_f1
Original KD SLM    0.5925         0.651795        0.5925  0.548398

Possible original KD prediction files:


In [16]:

import os
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from torch.utils.data import DataLoader

DATA_DIR = "/content/drive/MyDrive/Task1_Research"
ORIGINAL_KD_DIR = os.path.join(DATA_DIR, "distilled_student_model")
AUDIT_DIR = os.path.join(DATA_DIR, "original_kd_audit")
os.makedirs(AUDIT_DIR, exist_ok=True)

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "task1_test.csv")
)

assert os.path.isdir(ORIGINAL_KD_DIR), (
    f"Checkpoint not found: {ORIGINAL_KD_DIR}"
)

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer_kd = AutoTokenizer.from_pretrained(ORIGINAL_KD_DIR)
if tokenizer_kd.pad_token is None:
    tokenizer_kd.pad_token = tokenizer_kd.eos_token

bnb_config_kd = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading original KD base model...")
base_kd = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config_kd,
    device_map="auto"
)

print("Loading original KD adapter...")
kd_model = PeftModel.from_pretrained(
    base_kd,
    ORIGINAL_KD_DIR
)
kd_model.eval()
kd_model.config.use_cache = False

db_ids = tokenizer_kd.encode("DATABASE", add_special_tokens=False)
gen_ids = tokenizer_kd.encode("GENERAL", add_special_tokens=False)

assert len(db_ids) == 1 and len(gen_ids) == 1, (
    "Expected each class label to be one token."
)

class_ids = [db_ids[0], gen_ids[0]]

# Match the original KD student's raw-text evaluation setup
def make_batch(items):
    encoded = tokenizer_kd(
        [item for item in items],
        max_length=256,
        truncation=True,
        padding=True,
        return_tensors="pt"
    )
    return encoded

predictions = []
actual_labels = []
queries = []

BATCH_SIZE = 8

with torch.no_grad():
    for start in range(0, len(test_df), BATCH_SIZE):
        batch_df = test_df.iloc[start:start + BATCH_SIZE]
        texts = batch_df["text"].astype(str).tolist()

        batch = make_batch(texts)
        input_ids = batch["input_ids"].to("cuda")
        attention_mask = batch["attention_mask"].to("cuda")

        outputs = kd_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False
        )

        last_positions = attention_mask.sum(dim=1) - 1
        rows = torch.arange(input_ids.size(0), device="cuda")
        final_logits = outputs.logits[rows, last_positions, :]

        class_logits = final_logits[:, class_ids]
        batch_preds = class_logits.argmax(dim=1).cpu().tolist()

        predictions.extend(batch_preds)
        queries.extend(texts)
        actual_labels.extend(
            batch_df["label"].astype(str).str.upper().str.strip().tolist()
        )

label_names = ["DATABASE", "GENERAL"]
predicted_labels = [label_names[p] for p in predictions]

accuracy = accuracy_score(actual_labels, predicted_labels)

print(f"\nRe-evaluated test accuracy: {accuracy:.4%}")
print("\nClassification report:")
print(classification_report(
    actual_labels,
    predicted_labels,
    labels=label_names,
    digits=4,
    zero_division=0
))

cm = confusion_matrix(
    actual_labels,
    predicted_labels,
    labels=label_names
)
print("Confusion matrix (actual rows, predicted columns):")
print(pd.DataFrame(
    cm,
    index=["Actual DATABASE", "Actual GENERAL"],
    columns=["Predicted DATABASE", "Predicted GENERAL"]
))

# Save audited predictions separately
audit_df = pd.DataFrame({
    "text": queries,
    "actual_label": actual_labels,
    "predicted_label": predicted_labels,
    "correct": [
        actual == predicted
        for actual, predicted in zip(actual_labels, predicted_labels)
    ]
})

audit_df.to_csv(
    os.path.join(AUDIT_DIR, "original_kd_re_evaluation_predictions.csv"),
    index=False
)

pd.DataFrame([{
    "model": "Original KD SLM - re-evaluated",
    "test_accuracy": accuracy
}]).to_csv(
    os.path.join(AUDIT_DIR, "original_kd_re_evaluation_metrics.csv"),
    index=False
)

print("\nAudit outputs saved to:", AUDIT_DIR)


Loading original KD base model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading original KD adapter...

Re-evaluated test accuracy: 59.2500%

Classification report:
              precision    recall  f1-score   support

    DATABASE     0.5569    0.9050    0.6895       600
     GENERAL     0.7467    0.2800    0.4073       600

    accuracy                         0.5925      1200
   macro avg     0.6518    0.5925    0.5484      1200
weighted avg     0.6518    0.5925    0.5484      1200

Confusion matrix (actual rows, predicted columns):
                 Predicted DATABASE  Predicted GENERAL
Actual DATABASE                 543                 57
Actual GENERAL                  432                168

Audit outputs saved to: /content/drive/MyDrive/Task1_Research/original_kd_audit


In [17]:

import os
import pandas as pd
import numpy as np

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

soft_path = os.path.join(
    DATA_DIR,
    "task1_teacher_soft_targets_train.csv"
)

soft_df = pd.read_csv(soft_path)

print("Shape:", soft_df.shape)
print("Columns:", soft_df.columns.tolist())
print("\nFirst five rows:")
print(soft_df.head().to_string(index=False))

required = {
    "text", "label", "teacher_DATABASE", "teacher_GENERAL"
}
assert required.issubset(soft_df.columns), (
    f"Missing columns: {required - set(soft_df.columns)}"
)

db_prob = pd.to_numeric(
    soft_df["teacher_DATABASE"], errors="coerce"
)
gen_prob = pd.to_numeric(
    soft_df["teacher_GENERAL"], errors="coerce"
)

print("\nMissing soft targets:")
print("DATABASE:", int(db_prob.isna().sum()))
print("GENERAL:", int(gen_prob.isna().sum()))

print("\nSoft-target summary:")
print(pd.DataFrame({
    "teacher_DATABASE": db_prob,
    "teacher_GENERAL": gen_prob
}).describe().to_string())

print("\nProbability-sum summary:")
print((db_prob + gen_prob).describe().to_string())

print("\nRows with probabilities outside [0, 1]:")
invalid = (
    (db_prob < 0) | (db_prob > 1) |
    (gen_prob < 0) | (gen_prob > 1)
)
print(int(invalid.sum()))

print("\nTeacher's top class versus true label:")
valid = db_prob.notna() & gen_prob.notna()
teacher_label = np.where(
    db_prob[valid].to_numpy() >= gen_prob[valid].to_numpy(),
    "DATABASE",
    "GENERAL"
)
true_label = (
    soft_df.loc[valid, "label"]
    .astype(str).str.upper().str.strip().to_numpy()
)

print("Valid rows:", int(valid.sum()))
print("Teacher agreement with labels:", np.mean(teacher_label == true_label))
print(pd.crosstab(
    pd.Series(true_label, name="True label"),
    pd.Series(teacher_label, name="Teacher top class")
))


Shape: (4000, 4)
Columns: ['text', 'label', 'teacher_DATABASE', 'teacher_GENERAL']

First five rows:
                                              text    label  teacher_DATABASE  teacher_GENERAL
                            what is life all about  GENERAL          0.000013         0.999987
                         i want to listen to music  GENERAL          0.000008         0.999992
        What season has a Level of 2nd, 29 points? DATABASE          0.996406         0.003594
             how many calories does a big mac have  GENERAL          0.998590         0.001410
What was the latitude of the Clearwater townsship? DATABASE          0.998968         0.001032

Missing soft targets:
DATABASE: 0
GENERAL: 0

Soft-target summary:
       teacher_DATABASE  teacher_GENERAL
count       4000.000000      4000.000000
mean           0.492522         0.507478
std            0.472921         0.472921
min            0.000003         0.000016
25%            0.000082         0.002435
50%            

In [18]:

import os
import numpy as np
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

path = os.path.join(
    DATA_DIR,
    "finetuned_teacher_soft_targets_train.csv"
)

df = pd.read_csv(path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head().to_string(index=False))

prob_cols = ["teacher_DATABASE", "teacher_GENERAL"]
missing_cols = set(prob_cols + ["label"]) - set(df.columns)

if missing_cols:
    print("\nExpected columns not found:", missing_cols)
    print("Please send me the output; we will adapt the audit.")
else:
    probs = df[prob_cols].apply(pd.to_numeric, errors="coerce")
    labels = df["label"].astype(str).str.upper().str.strip()

    print("\nMissing probabilities:")
    print(probs.isna().sum())

    print("\nProbability sum statistics:")
    print((probs["teacher_DATABASE"] + probs["teacher_GENERAL"]).describe())

    valid = probs.notna().all(axis=1)
    valid &= probs.ge(0).all(axis=1)
    valid &= probs.le(1).all(axis=1)

    teacher_pred = np.where(
        probs.loc[valid, "teacher_DATABASE"].to_numpy()
        >= probs.loc[valid, "teacher_GENERAL"].to_numpy(),
        "DATABASE",
        "GENERAL"
    )

    true_labels = labels.loc[valid].to_numpy()

    print("\nValid rows:", int(valid.sum()))
    print(
        "Teacher agreement with labels:",
        round(float(np.mean(teacher_pred == true_labels)), 6)
    )

    print("\nAgreement by true class:")
    print(pd.crosstab(
        pd.Series(true_labels, name="True label"),
        pd.Series(teacher_pred, name="Teacher prediction")
    ))

    print("\nProbability summary:")
    print(probs.describe().to_string())


Shape: (4000, 4)
Columns: ['text', 'label', 'teacher_DATABASE', 'teacher_GENERAL']

First 5 rows:
                                                                                                                  text    label  teacher_DATABASE  teacher_GENERAL
                                                                                       how many coupless totalled 14.5 DATABASE      1.000000e+00     2.103088e-18
                                                                         what are the atmospheric conditions in sparks  GENERAL      1.228186e-10     1.000000e+00
Can you tell me the highest Total votes that has the % of popular vote of 0.86%, and the # of seats won larger than 0? DATABASE      1.000000e+00     3.289783e-17
                                                                              what can you tell me about miami weather  GENERAL      4.534574e-13     1.000000e+00
                                                         i am needing to know the curre

In [19]:

import re
import pandas as pd

def normalize_query(text):
    text = str(text).lower().strip()
    return re.sub(r"\s+", " ", text)

splits = {
    "train": train_df,
    "validation": val_df,
    "test": test_df
}

normalized = {
    name: set(df["text"].map(normalize_query))
    for name, df in splits.items()
}

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    overlap = normalized[a] & normalized[b]
    print(f"{a} vs {b}: {len(overlap)} normalized exact overlaps")

print("\nUnique normalized queries:")
for name, values in normalized.items():
    print(name, len(values))


train vs validation: 0 normalized exact overlaps
train vs test: 0 normalized exact overlaps
validation vs test: 0 normalized exact overlaps

Unique normalized queries:
train 4000
validation 799
test 1200


In [20]:

import os
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

DATA_DIR = "/content/drive/MyDrive/Task1_Research"
TEACHER_DIR = os.path.join(DATA_DIR, "finetuned_teacher_7b")

assert os.path.isdir(TEACHER_DIR), (
    f"Teacher checkpoint not found: {TEACHER_DIR}"
)

val_df = pd.read_csv(
    os.path.join(DATA_DIR, "task1_validation.csv")
)

teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_DIR)

if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token

teacher_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading 7B base model...")
teacher_base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=teacher_bnb_config,
    device_map="auto"
)

print("Loading fine-tuned teacher adapter...")
teacher = PeftModel.from_pretrained(
    teacher_base,
    TEACHER_DIR
)
teacher.eval()
teacher.config.use_cache = False

db_ids = teacher_tokenizer.encode(
    "DATABASE", add_special_tokens=False
)
gen_ids = teacher_tokenizer.encode(
    "GENERAL", add_special_tokens=False
)

assert len(db_ids) == 1 and len(gen_ids) == 1

class_ids = [db_ids[0], gen_ids[0]]
predictions = []
actual = []

BATCH_SIZE = 2

with torch.no_grad():
    for start in range(0, len(val_df), BATCH_SIZE):
        batch_df = val_df.iloc[start:start + BATCH_SIZE]
        texts = batch_df["text"].astype(str).tolist()

        batch = teacher_tokenizer(
            texts,
            max_length=256,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )

        input_ids = batch["input_ids"].to("cuda")
        attention_mask = batch["attention_mask"].to("cuda")

        outputs = teacher(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False
        )

        last_positions = attention_mask.sum(dim=1) - 1
        rows = torch.arange(input_ids.size(0), device="cuda")
        final_logits = outputs.logits[rows, last_positions, :]

        class_logits = final_logits[:, class_ids]
        batch_preds = class_logits.argmax(dim=1).cpu().tolist()

        predictions.extend(batch_preds)
        actual.extend(
            batch_df["label"].astype(str).str.upper().str.strip().tolist()
        )

labels = ["DATABASE", "GENERAL"]
predicted = [labels[p] for p in predictions]

print(f"\nTeacher validation accuracy: {accuracy_score(actual, predicted):.4%}")
print("\nClassification report:")
print(classification_report(
    actual,
    predicted,
    labels=labels,
    digits=4,
    zero_division=0
))

print("Confusion matrix (actual rows, predicted columns):")
print(pd.DataFrame(
    confusion_matrix(actual, predicted, labels=labels),
    index=["Actual DATABASE", "Actual GENERAL"],
    columns=["Predicted DATABASE", "Predicted GENERAL"]
))

out = val_df.copy()
out["predicted_label"] = predicted
out["correct"] = (
    out["label"].astype(str).str.upper().str.strip()
    == out["predicted_label"]
)
out.to_csv(
    os.path.join(DATA_DIR, "finetuned_teacher_validation_audit.csv"),
    index=False
)

print("\nSaved teacher validation audit.")


Loading 7B base model...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loading fine-tuned teacher adapter...

Teacher validation accuracy: 37.1715%

Classification report:
              precision    recall  f1-score   support

    DATABASE     0.3862    0.4325    0.4080       400
     GENERAL     0.3533    0.3108    0.3307       399

    accuracy                         0.3717       799
   macro avg     0.3697    0.3716    0.3693       799
weighted avg     0.3697    0.3717    0.3694       799

Confusion matrix (actual rows, predicted columns):
                 Predicted DATABASE  Predicted GENERAL
Actual DATABASE                 173                227
Actual GENERAL                  275                124

Saved teacher validation audit.


In [21]:

import os
import json

DATA_DIR = "/content/drive/MyDrive/Task1_Research"
TEACHER_DIR = os.path.join(DATA_DIR, "finetuned_teacher_7b")

for filename in ["adapter_config.json", "config.json"]:
    path = os.path.join(TEACHER_DIR, filename)
    print(f"\n--- {filename} ---")

    if not os.path.exists(path):
        print("Not found at teacher folder root.")
        continue

    with open(path, "r") as f:
        config = json.load(f)

    keys_to_show = [
        "base_model_name_or_path",
        "task_type",
        "peft_type",
        "architectures",
        "model_type",
        "num_labels",
        "id2label",
        "label2id"
    ]

    for key in keys_to_show:
        if key in config:
            print(f"{key}: {config[key]}")

print("\n--- Adapter files ---")
if os.path.isdir(TEACHER_DIR):
    for filename in os.listdir(TEACHER_DIR):
        print(filename)



--- adapter_config.json ---
base_model_name_or_path: Qwen/Qwen2.5-7B-Instruct
task_type: SEQ_CLS
peft_type: LORA

--- config.json ---
Not found at teacher folder root.

--- Adapter files ---
tokenizer.json
checkpoint-1000
checkpoint-1500
README.md
adapter_config.json
tokenizer_config.json
chat_template.jinja
training_args.bin
adapter_model.safetensors


In [22]:

import os
import json
import torch

DATA_DIR = "/content/drive/MyDrive/Task1_Research"
TEACHER_DIR = os.path.join(DATA_DIR, "finetuned_teacher_7b")

# Inspect LoRA adapter configuration
adapter_path = os.path.join(TEACHER_DIR, "adapter_config.json")
with open(adapter_path, "r") as f:
    adapter_config = json.load(f)

print("=== Adapter configuration ===")
for key in [
    "base_model_name_or_path",
    "task_type",
    "target_modules",
    "r",
    "lora_alpha",
    "lora_dropout",
    "modules_to_save"
]:
    print(f"{key}: {adapter_config.get(key)}")

# Inspect saved Hugging Face training arguments
args_path = os.path.join(TEACHER_DIR, "training_args.bin")
print("\n=== Training arguments ===")

if os.path.exists(args_path):
    try:
        args = torch.load(
            args_path,
            map_location="cpu",
            weights_only=False
        )
        for key in [
            "output_dir",
            "num_train_epochs",
            "learning_rate",
            "per_device_train_batch_size",
            "gradient_accumulation_steps",
            "max_steps",
            "save_steps",
            "eval_strategy",
            "evaluation_strategy",
            "label_names"
        ]:
            print(f"{key}: {getattr(args, key, None)}")
    except Exception as e:
        print("Could not read training arguments:", repr(e))
else:
    print("training_args.bin not found")

print("\n=== Checkpoint folders ===")
for name in sorted(os.listdir(TEACHER_DIR)):
    path = os.path.join(TEACHER_DIR, name)
    if os.path.isdir(path) and name.startswith("checkpoint-"):
        print(name, "contains:", os.listdir(path))


=== Adapter configuration ===
base_model_name_or_path: Qwen/Qwen2.5-7B-Instruct
task_type: SEQ_CLS
target_modules: ['down_proj', 'o_proj', 'k_proj', 'q_proj', 'up_proj', 'gate_proj', 'v_proj']
r: 8
lora_alpha: 16
lora_dropout: 0.05
modules_to_save: ['score', 'classifier', 'score']

=== Training arguments ===
output_dir: /content/drive/MyDrive/Task1_Research/finetuned_teacher_7b
num_train_epochs: 3
learning_rate: 0.0002
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
max_steps: -1
save_steps: 500
eval_strategy: IntervalStrategy.EPOCH
evaluation_strategy: None
label_names: None

=== Checkpoint folders ===
checkpoint-1000 contains: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'scaler.pt', 'rng_state.pth', 'trainer_state.json']
checkpoint-1500 contains: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja

In [24]:

import os
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig
)
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

DATA_DIR = "/content/drive/MyDrive/Task1_Research"
TEACHER_DIR = os.path.join(DATA_DIR, "finetuned_teacher_7b")

val_df = pd.read_csv(
    os.path.join(DATA_DIR, "task1_validation.csv")
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(TEACHER_DIR)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading sequence-classification base model...")
base = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

# Ensure label index interpretation is explicit
base.config.id2label = {0: "DATABASE", 1: "GENERAL"}
base.config.label2id = {"DATABASE": 0, "GENERAL": 1}

print("Loading saved teacher adapter and classification head...")
teacher = PeftModel.from_pretrained(base, TEACHER_DIR)
teacher.eval()

predictions = []
actual = []

BATCH_SIZE = 2

with torch.no_grad():
    for start in range(0, len(val_df), BATCH_SIZE):
        batch_df = val_df.iloc[start:start + BATCH_SIZE]
        texts = batch_df["text"].astype(str).tolist()

        batch = tokenizer(
            texts,
            max_length=256,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )

        batch = {k: v.to("cuda") for k, v in batch.items()}
        outputs = teacher(**batch)

        preds = outputs.logits.argmax(dim=-1).cpu().tolist()
        predictions.extend(preds)

        actual.extend(
            batch_df["label"].astype(str).str.upper().str.strip().tolist()
        )

label_names = ["DATABASE", "GENERAL"]
predicted = [label_names[p] for p in predictions]

print(
    f"\nSequence-classification validation accuracy: "
    f"{accuracy_score(actual, predicted):.4%}"
)

print("\nClassification report:")
print(classification_report(
    actual,
    predicted,
    labels=label_names,
    digits=4,
    zero_division=0
))

print("Confusion matrix (actual rows, predicted columns):")
print(pd.DataFrame(
    confusion_matrix(actual, predicted, labels=label_names),
    index=["Actual DATABASE", "Actual GENERAL"],
    columns=["Predicted DATABASE", "Predicted GENERAL"]
))

audit = val_df.copy()
audit["predicted_label"] = predicted
audit["correct"] = (
    audit["label"].astype(str).str.upper().str.strip()
    == audit["predicted_label"]
)

audit_path = os.path.join(
    DATA_DIR, "finetuned_teacher_validation_seqcls_audit.csv"
)
audit.to_csv(audit_path, index=False)
print("\nSaved audit to:", audit_path)


Loading tokenizer...
Loading sequence-classification base model...


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [25]:

!nvidia-smi


Sun Oct 18 11:00:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             33W /   70W |   11799MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [26]:

import torch
from transformers import (
    AutoModelForSequenceClassification,
    BitsAndBytesConfig
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    num_labels=2,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16
)

print("Model loaded on GPU successfully.")


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.81 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.24 GiB is free. Including non-PyTorch memory, this process has 11.32 GiB memory in use. Of the allocated memory 11.13 GiB is allocated by PyTorch, and 45.47 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [2]:

from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:

import os

DATA_DIR = "/content/drive/MyDrive/Task1_Research"
TEACHER_DIR = os.path.join(DATA_DIR, "finetuned_teacher_7b")

print("Research folder exists:", os.path.isdir(DATA_DIR))
print("Teacher folder exists:", os.path.isdir(TEACHER_DIR))

assert os.path.isdir(TEACHER_DIR), "Teacher folder not found!"

print("\nTeacher folder contents:")
print(os.listdir(TEACHER_DIR))


Research folder exists: True
Teacher folder exists: True

Teacher folder contents:
['tokenizer.json', 'checkpoint-1000', 'checkpoint-1500', 'README.md', 'adapter_config.json', 'tokenizer_config.json', 'chat_template.jinja', 'training_args.bin', 'adapter_model.safetensors']


In [4]:

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TEACHER_DIR)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully.")
print("Padding token:", tokenizer.pad_token)


Tokenizer loaded successfully.
Padding token: <|endoftext|>


In [5]:

import torch
from transformers import (
    AutoModelForSequenceClassification,
    BitsAndBytesConfig
)

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("\nLoading 7B sequence-classification base model...")

base = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory={0: "12GiB", "cpu": "24GiB"},
    torch_dtype=torch.float16
)

print("\nBase model loaded successfully.")


PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4

Loading 7B sequence-classification base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-7B-Instruct
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Base model loaded successfully.


In [6]:

from peft import PeftModel
import torch

print("Loading saved fine-tuned teacher adapter...")

teacher = PeftModel.from_pretrained(
    base,
    TEACHER_DIR,
    is_trainable=False
)

teacher.eval()

print("Teacher adapter loaded.")
print("Model class:", type(teacher).__name__)
print("Classification configuration:")
print("Number of labels:", teacher.config.num_labels)
print("Label mapping:", teacher.config.id2label)

print("\nGPU memory allocated:")
print(f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB")


Loading saved fine-tuned teacher adapter...
Teacher adapter loaded.
Model class: PeftModelForSequenceClassification
Classification configuration:
Number of labels: 2
Label mapping: {0: 'LABEL_0', 1: 'LABEL_1'}

GPU memory allocated:
4.24 GiB


In [8]:


# Fix padding configuration without reloading the model
tokenizer.padding_side = "right"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

teacher.config.pad_token_id = tokenizer.pad_token_id
teacher.config.get_text_config().pad_token_id = tokenizer.pad_token_id

print("Tokenizer pad token:", tokenizer.pad_token)
print("Tokenizer pad token ID:", tokenizer.pad_token_id)
print("Model pad token ID:", teacher.config.pad_token_id)
print("Padding configuration updated.")


Tokenizer pad token: <|endoftext|>
Tokenizer pad token ID: 151643
Model pad token ID: 151643
Padding configuration updated.


In [9]:

import pandas as pd
import torch

all_predictions = []
BATCH_SIZE = 2

teacher.eval()

with torch.no_grad():
    for start in range(0, len(val_df), BATCH_SIZE):
        texts = val_df["text"].astype(str).iloc[
            start:start + BATCH_SIZE
        ].tolist()

        batch = tokenizer(
            texts,
            max_length=256,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )

        batch = {k: v.to("cuda") for k, v in batch.items()}
        outputs = teacher(**batch)

        preds = outputs.logits.argmax(dim=-1).cpu().tolist()
        all_predictions.extend(preds)

print("Validation predictions generated:", len(all_predictions))
print("\nPredicted class-index distribution:")
print(pd.Series(all_predictions).value_counts().sort_index())

print("\nFirst 20 pairs (true label, predicted index):")
for true_label, pred_index in zip(
    val_df["label"].head(20), all_predictions[:20]
):
    print(repr(true_label), "->", pred_index)


Validation predictions generated: 799

Predicted class-index distribution:
0    400
1    399
Name: count, dtype: int64

First 20 pairs (true label, predicted index):
'GENERAL' -> 1
'GENERAL' -> 1
'DATABASE' -> 0
'GENERAL' -> 1
'DATABASE' -> 0
'GENERAL' -> 1
'DATABASE' -> 0
'GENERAL' -> 1
'GENERAL' -> 1
'GENERAL' -> 1
'DATABASE' -> 0
'DATABASE' -> 0
'GENERAL' -> 1
'DATABASE' -> 0
'DATABASE' -> 0
'DATABASE' -> 0
'DATABASE' -> 0
'DATABASE' -> 0
'GENERAL' -> 1
'GENERAL' -> 1


In [10]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
import pandas as pd
import os

# Verify that the observed mapping is consistent with training labels.
# The earlier teacher-training code must ultimately confirm this mapping.
label_map = {0: "DATABASE", 1: "GENERAL"}

predicted_labels = [label_map[p] for p in all_predictions]
actual_labels = (
    val_df["label"].astype(str).str.upper().str.strip().tolist()
)

print("Validation accuracy:",
      f"{accuracy_score(actual_labels, predicted_labels):.4%}")

print("\nClassification report:")
print(classification_report(
    actual_labels,
    predicted_labels,
    labels=["DATABASE", "GENERAL"],
    digits=4,
    zero_division=0
))

print("Confusion matrix (actual rows, predicted columns):")
print(pd.DataFrame(
    confusion_matrix(
        actual_labels,
        predicted_labels,
        labels=["DATABASE", "GENERAL"]
    ),
    index=["Actual DATABASE", "Actual GENERAL"],
    columns=["Predicted DATABASE", "Predicted GENERAL"]
))

# Save this diagnostic separately; do not overwrite earlier results.
audit_df = val_df.copy()
audit_df["predicted_label"] = predicted_labels
audit_df["correct"] = [
    actual == predicted
    for actual, predicted in zip(actual_labels, predicted_labels)
]

save_path = os.path.join(
    DATA_DIR,
    "finetuned_teacher_validation_seqcls_audit.csv"
)
audit_df.to_csv(save_path, index=False)

print("\nSaved:", save_path)


Validation accuracy: 100.0000%

Classification report:
              precision    recall  f1-score   support

    DATABASE     1.0000    1.0000    1.0000       400
     GENERAL     1.0000    1.0000    1.0000       399

    accuracy                         1.0000       799
   macro avg     1.0000    1.0000    1.0000       799
weighted avg     1.0000    1.0000    1.0000       799

Confusion matrix (actual rows, predicted columns):
                 Predicted DATABASE  Predicted GENERAL
Actual DATABASE                 400                  0
Actual GENERAL                    0                399

Saved: /content/drive/MyDrive/Task1_Research/finetuned_teacher_validation_seqcls_audit.csv


In [11]:

import os
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. Load the test dataset
DATA_DIR = "/content/drive/MyDrive/Task1_Research"
TEST_PATH = os.path.join(DATA_DIR, "task1_test.csv")

test_df = pd.read_csv(TEST_PATH)

print("Test dataset shape:", test_df.shape)
print("\nActual label distribution:")
print(test_df["label"].value_counts())

# 2. Confirm the label mapping used during training
# Use this mapping only if your training code used:
# DATABASE = 0, GENERAL = 1
label_map = {
    0: "DATABASE",
    1: "GENERAL"
}

# 3. Generate predictions using the loaded SEQ_CLS teacher
all_predictions = []
BATCH_SIZE = 2

teacher.eval()

with torch.no_grad():
    for start in range(0, len(test_df), BATCH_SIZE):
        texts = (
            test_df["text"]
            .astype(str)
            .iloc[start:start + BATCH_SIZE]
            .tolist()
        )

        batch = tokenizer(
            texts,
            max_length=256,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )

        # Your existing model was loaded onto CUDA
        batch = {k: v.to("cuda") for k, v in batch.items()}

        outputs = teacher(**batch)
        predictions = outputs.logits.argmax(dim=-1).cpu().tolist()

        all_predictions.extend(predictions)

print("\nTotal predictions:", len(all_predictions))

# 4. Convert predicted indices to class names
y_true = test_df["label"].astype(str).tolist()
y_pred = [label_map[p] for p in all_predictions]

# 5. Calculate metrics
accuracy = accuracy_score(y_true, y_pred)

print(f"\nFine-tuned teacher TEST accuracy: {accuracy:.4%}")

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=["DATABASE", "GENERAL"],
    digits=4,
    zero_division=0
))

print("\nConfusion matrix (actual rows, predicted columns):")
print(pd.DataFrame(
    confusion_matrix(
        y_true,
        y_pred,
        labels=["DATABASE", "GENERAL"]
    ),
    index=["Actual DATABASE", "Actual GENERAL"],
    columns=["Predicted DATABASE", "Predicted GENERAL"]
))

# 6. Save predictions separately
predictions_path = os.path.join(
    DATA_DIR,
    "finetuned_teacher_test_predictions_audit.csv"
)

results_df = test_df.copy()
results_df["teacher_predicted_label"] = y_pred
results_df["correct"] = (
    results_df["label"].astype(str)
    == results_df["teacher_predicted_label"]
)

results_df.to_csv(predictions_path, index=False)

print("\nSaved predictions to:", predictions_path)

# 7. Save test metrics separately
metrics_path = os.path.join(
    DATA_DIR,
    "finetuned_teacher_test_metrics_audit.csv"
)

pd.DataFrame([{
    "model": "Fine-tuned Qwen2.5-7B Sequence Classification Teacher",
    "test_samples": len(test_df),
    "correct_predictions": int((results_df["correct"]).sum()),
    "accuracy": accuracy
}]).to_csv(metrics_path, index=False)

print("Saved metrics to:", metrics_path)


Test dataset shape: (1200, 2)

Actual label distribution:
label
GENERAL     600
DATABASE    600
Name: count, dtype: int64

Total predictions: 1200

Fine-tuned teacher TEST accuracy: 99.8333%

Classification report:
              precision    recall  f1-score   support

    DATABASE     0.9983    0.9983    0.9983       600
     GENERAL     0.9983    0.9983    0.9983       600

    accuracy                         0.9983      1200
   macro avg     0.9983    0.9983    0.9983      1200
weighted avg     0.9983    0.9983    0.9983      1200


Confusion matrix (actual rows, predicted columns):
                 Predicted DATABASE  Predicted GENERAL
Actual DATABASE                 599                  1
Actual GENERAL                    1                599

Saved predictions to: /content/drive/MyDrive/Task1_Research/finetuned_teacher_test_predictions_audit.csv
Saved metrics to: /content/drive/MyDrive/Task1_Research/finetuned_teacher_test_metrics_audit.csv


In [12]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

# Existing test prediction files
teacher_path = os.path.join(
    DATA_DIR,
    "finetuned_teacher_test_predictions_audit.csv"
)

student_path = os.path.join(
    DATA_DIR,
    "finetuned_teacher_kd_student_test_predictions.csv"
)

# Check that both files exist
for path in [teacher_path, student_path]:
    print(f"\nFile: {os.path.basename(path)}")
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        print("Size:", os.path.getsize(path), "bytes")

# Load files
if os.path.exists(teacher_path) and os.path.exists(student_path):
    teacher_df = pd.read_csv(teacher_path)
    student_df = pd.read_csv(student_path)

    print("\n========== TEACHER FILE ==========")
    print("Shape:", teacher_df.shape)
    print("Columns:", teacher_df.columns.tolist())
    print(teacher_df.head(5).to_string(index=False))

    print("\n========== KD STUDENT FILE ==========")
    print("Shape:", student_df.shape)
    print("Columns:", student_df.columns.tolist())
    print(student_df.head(5).to_string(index=False))
else:
    print(
        "\nOne or both files are missing. "
        "Check the filenames in your Google Drive folder."
    )



File: finetuned_teacher_test_predictions_audit.csv
Exists: True
Size: 85160 bytes

File: finetuned_teacher_kd_student_test_predictions.csv
Exists: True
Size: 84056 bytes

========== TEACHER FILE ==========
Shape: (1200, 4)
Columns: ['text', 'label', 'teacher_predicted_label', 'correct']
                                                                                                                    text    label teacher_predicted_label  correct
                                                                                   how many calories does an orange have  GENERAL                 GENERAL     True
                                                                                                      let's hear a joke!  GENERAL                DATABASE    False
What is the sum of the pick numbers where the player was picked in round 5 and overall pick number was greater than 159? DATABASE                DATABASE     True
                                                           

In [13]:

import os
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

teacher_path = os.path.join(
    DATA_DIR, "finetuned_teacher_test_predictions_audit.csv"
)
student_path = os.path.join(
    DATA_DIR, "finetuned_teacher_kd_student_test_predictions.csv"
)

teacher_df = pd.read_csv(teacher_path)
student_df = pd.read_csv(student_path)

# 1. Check row-by-row alignment
same_length = len(teacher_df) == len(student_df)
same_text_order = (
    teacher_df["text"].astype(str).tolist()
    == student_df["text"].astype(str).tolist()
)
same_labels = (
    teacher_df["label"].astype(str).tolist()
    == student_df["label"].astype(str).tolist()
)

print("========== DATA ALIGNMENT ==========")
print("Same number of rows:", same_length)
print("Same query order:", same_text_order)
print("Same ground-truth labels:", same_labels)

# 2. Independently recompute accuracy
teacher_accuracy = accuracy_score(
    teacher_df["label"].astype(str),
    teacher_df["teacher_predicted_label"].astype(str)
)
student_accuracy = accuracy_score(
    student_df["label"].astype(str),
    student_df["predicted_label"].astype(str)
)

print("\n========== RECOMPUTED ACCURACY ==========")
print(f"Fine-tuned teacher: {teacher_accuracy:.4%}")
print(f"Fine-tuned teacher + KD student: {student_accuracy:.4%}")

# 3. Check the teacher's saved 'correct' column
teacher_correct_check = (
    teacher_df["label"].astype(str)
    == teacher_df["teacher_predicted_label"].astype(str)
)
print(
    "\nTeacher 'correct' column consistent:",
    teacher_correct_check.equals(teacher_df["correct"].astype(bool))
)

# 4. Compare models only if test rows and labels align
if same_length and same_text_order and same_labels:
    comparison = pd.DataFrame({
        "text": teacher_df["text"],
        "label": teacher_df["label"],
        "teacher_prediction": teacher_df["teacher_predicted_label"],
        "student_prediction": student_df["predicted_label"]
    })

    comparison["teacher_correct"] = (
        comparison["label"] == comparison["teacher_prediction"]
    )
    comparison["student_correct"] = (
        comparison["label"] == comparison["student_prediction"]
    )

    teacher_only_wrong = (
        ~comparison["teacher_correct"]
        & comparison["student_correct"]
    )
    student_only_wrong = (
        comparison["teacher_correct"]
        & ~comparison["student_correct"]
    )
    both_wrong = (
        ~comparison["teacher_correct"]
        & ~comparison["student_correct"]
    )

    print("\n========== DISAGREEMENT AUDIT ==========")
    print("Teacher wrong, student correct:", int(teacher_only_wrong.sum()))
    print("Teacher correct, student wrong:", int(student_only_wrong.sum()))
    print("Both models wrong:", int(both_wrong.sum()))
    print(
        "Both models correct:",
        int((comparison["teacher_correct"] & comparison["student_correct"]).sum())
    )

    print("\n========== TEACHER-ONLY ERRORS ==========")
    print(comparison[teacher_only_wrong].to_string(index=False))

    print("\n========== STUDENT-ONLY ERRORS ==========")
    print(comparison[student_only_wrong].to_string(index=False))

    print("\n========== BOTH MODELS WRONG ==========")
    print(comparison[both_wrong].to_string(index=False))

    print("\n========== STUDENT CONFUSION MATRIX ==========")
    print(pd.DataFrame(
        confusion_matrix(
            comparison["label"],
            comparison["student_prediction"],
            labels=["DATABASE", "GENERAL"]
        ),
        index=["Actual DATABASE", "Actual GENERAL"],
        columns=["Predicted DATABASE", "Predicted GENERAL"]
    ))
else:
    print(
        "\nSTOP: The files are not aligned. "
        "Do not compare their predictions until we investigate."
    )


========== DATA ALIGNMENT ==========
Same number of rows: True
Same query order: True
Same ground-truth labels: True

========== RECOMPUTED ACCURACY ==========
Fine-tuned teacher: 99.8333%
Fine-tuned teacher + KD student: 99.9167%

Teacher 'correct' column consistent: True

========== DISAGREEMENT AUDIT ==========
Teacher wrong, student correct: 1
Teacher correct, student wrong: 0
Both models wrong: 1
Both models correct: 1198

========== TEACHER-ONLY ERRORS ==========
              text   label teacher_prediction student_prediction  teacher_correct  student_correct
let's hear a joke! GENERAL           DATABASE            GENERAL            False             True

========== STUDENT-ONLY ERRORS ==========
Empty DataFrame
Columns: [text, label, teacher_prediction, student_prediction, teacher_correct, student_correct]
Index: []

========== BOTH MODELS WRONG ==========
                               text    label teacher_prediction student_prediction  teacher_correct  student_correct
what

In [14]:

import pandas as pd
import os

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "task1_test.csv")
)

query = "what is the spanish word for garden"

matches = test_df[
    test_df["text"].astype(str).str.strip().str.lower()
    == query
]

print("Matching test examples:")
print(matches.to_string(index=False))

print("\nExact query occurrences in the training split:")
train_df = pd.read_csv(
    os.path.join(DATA_DIR, "task1_train.csv")
)

train_matches = train_df[
    train_df["text"].astype(str).str.strip().str.lower()
    == query
]

print("Count:", len(train_matches))
if len(train_matches):
    print(train_matches.to_string(index=False))

print("\nExact query occurrences in the validation split:")
val_df = pd.read_csv(
    os.path.join(DATA_DIR, "task1_validation.csv")
)

val_matches = val_df[
    val_df["text"].astype(str).str.strip().str.lower()
    == query
]

print("Count:", len(val_matches))
if len(val_matches):
    print(val_matches.to_string(index=False))


Matching test examples:
                               text    label
what is the spanish word for garden DATABASE

Exact query occurrences in the training split:
Count: 0

Exact query occurrences in the validation split:
Count: 0


In [15]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

files = {
    "TRAIN": "task1_train.csv",
    "VALIDATION": "task1_validation.csv",
    "TEST": "task1_test.csv"
}

keywords = ["spanish", "garden", "word for"]

for split, filename in files.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))

    # Search broadly for related wording
    mask = df["text"].astype(str).str.contains(
        "|".join(keywords),
        case=False,
        regex=True,
        na=False
    )

    print(f"\n========== {split} MATCHES ==========")
    print("Number of matches:", int(mask.sum()))

    if mask.any():
        print(
            df.loc[mask, ["text", "label"]]
            .head(30)
            .to_string(index=False)
        )



========== TRAIN MATCHES ==========
Number of matches: 27
                                                                                            text    label
                                                      can you translate milk into spanish for me  GENERAL
            what is the team when the location attendace is boston garden and the series is 0-1? DATABASE
                                                              how can i say thank you in spanish  GENERAL
                                                                       what is spanish for hello  GENERAL
                                                                   how do you say cat in spanish  GENERAL
                                                             tell me how to say hello in spanish  GENERAL
                                               how do they say "where's the bathroom" in spanish  GENERAL
                                                        can you translate hi into spanish for

In [16]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

teacher_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "finetuned_teacher_test_predictions_audit.csv"
    )
)

student_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "finetuned_teacher_kd_student_test_predictions.csv"
    )
)

# Merge predictions by row position after the alignment audit
audit = pd.DataFrame({
    "text": teacher_df["text"],
    "true_label": teacher_df["label"],
    "teacher_prediction": teacher_df["teacher_predicted_label"],
    "student_prediction": student_df["predicted_label"]
})

audit["teacher_wrong"] = (
    audit["true_label"] != audit["teacher_prediction"]
)
audit["student_wrong"] = (
    audit["true_label"] != audit["student_prediction"]
)

errors = audit[audit["teacher_wrong"] | audit["student_wrong"]]

print("Number of test examples with at least one model error:", len(errors))
print(errors.to_string(index=False))


Number of test examples with at least one model error: 2
                               text true_label teacher_prediction student_prediction  teacher_wrong  student_wrong
                 let's hear a joke!    GENERAL           DATABASE            GENERAL           True          False
what is the spanish word for garden   DATABASE            GENERAL            GENERAL           True           True


In [17]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

for split in ["train", "validation", "test"]:
    path = os.path.join(DATA_DIR, f"task1_{split}.csv")
    df = pd.read_csv(path)

    # Focus on translation / language-intent examples
    mask = df["text"].astype(str).str.contains(
        r"spanish|translate|word for|say .* in |in spanish",
        case=False,
        regex=True,
        na=False
    )

    matches = df.loc[mask, ["text", "label"]]

    print(f"\n{split.upper()}: {len(matches)} related examples")
    print(matches.head(40).to_string(index=False))



TRAIN: 74 related examples
                                                                                            text    label
                                                      can you translate milk into spanish for me  GENERAL
                                                                how should i say hello in french  GENERAL
                                                                how do they say hello in germany  GENERAL
                                                                   how do they say yes in brazil  GENERAL
                                                                 how would i say hello in french  GENERAL
                                                              how can i say thank you in spanish  GENERAL
                                                                       what is spanish for hello  GENERAL
                                                                   how do you say cat in spanish  GENERAL
                  

In [18]:

import os

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

print("Research folder files:\n")

for filename in sorted(os.listdir(DATA_DIR)):
    lower_name = filename.lower()

    if any(term in lower_name for term in [
        "dataset", "source", "preprocess", "clean",
        "label", "split", "audit", "readme", "task1"
    ]):
        print(filename)


Research folder files:

finetuned_teacher_test_metrics_audit.csv
finetuned_teacher_test_predictions_audit.csv
finetuned_teacher_validation_audit.csv
finetuned_teacher_validation_seqcls_audit.csv
native_student_validation_audit.csv
original_kd_audit
task1_baseline_metrics.csv
task1_data_leakage_audit.csv
task1_label_audit_sample.csv
task1_labeling_rules.txt
task1_manual_review_candidates.csv
task1_teacher_soft_targets_checkpoint.csv
task1_teacher_soft_targets_pilot.csv
task1_teacher_soft_targets_train.csv
task1_test.csv
task1_tfidf_logistic_baseline.joblib
task1_train.csv
task1_train_validation_review.csv
task1_validation.csv


In [19]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

# 1. Read the documented labeling rules
rules_path = os.path.join(DATA_DIR, "task1_labeling_rules.txt")

print("=" * 60)
print("TASK 1 LABELING RULES")
print("=" * 60)

if os.path.exists(rules_path):
    with open(rules_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("Labeling rules file not found.")

# 2. Check whether the suspicious query is in the manual-review file
review_path = os.path.join(
    DATA_DIR, "task1_manual_review_candidates.csv"
)

print("\n" + "=" * 60)
print("MANUAL REVIEW CANDIDATES")
print("=" * 60)

if os.path.exists(review_path):
    review_df = pd.read_csv(review_path)

    print("File shape:", review_df.shape)
    print("Columns:", review_df.columns.tolist())

    # Search every text column for the garden query
    text_columns = review_df.select_dtypes(
        include=["object", "string"]
    ).columns

    mask = pd.Series(False, index=review_df.index)

    for col in text_columns:
        mask |= review_df[col].astype(str).str.contains(
            "what is the spanish word for garden",
            case=False,
            regex=False,
            na=False
        )

    if mask.any():
        print("\nMatching review candidates:")
        print(review_df.loc[mask].to_string(index=False))
    else:
        print("\nGarden query not found in this review file.")

    print("\nFirst 15 review candidates:")
    print(review_df.head(15).to_string(index=False))
else:
    print("Manual-review candidates file not found.")


TASK 1 LABELING RULES

Task 1: Semantic Query Classification — Labeling Rules

DATABASE:
- Requests to write SQL queries.
- Requests to retrieve or filter structured database records.
- Requests to count, compare, join, or aggregate database records.
- Requests to insert, update, or delete database records.
- Natural-language questions intended to query structured records.

GENERAL:
- Greetings and ordinary conversation.
- Jokes, translations, explanations, and general advice.
- General factual questions without a clear structured-record query intent.

REVIEW:
- Ambiguous queries whose intent cannot be determined confidently.
- Malformed or incomplete queries that cannot be interpreted reliably.

Important:
- REVIEW is for manual inspection, not a final training class.
- Do not change original labels automatically based only on teacher predictions.
- Keep original datasets unchanged.
- Keep the final test labels independent from teacher-generated predictions.


MANUAL REVIEW CANDIDATES

In [20]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

review_path = os.path.join(
    DATA_DIR,
    "task1_test_label_review_audit.csv"
)

review_df = pd.DataFrame([{
    "split": "test",
    "text": "what is the spanish word for garden",
    "original_label": "DATABASE",
    "suggested_label": "GENERAL",
    "review_status": "Pending manual confirmation",
    "reason": (
        "The query appears to request a translation. "
        "The documented labeling rules classify translations "
        "as GENERAL. Similar translation queries in the splits "
        "are also labeled GENERAL."
    ),
    "official_test_label_changed": False
}])

review_df.to_csv(review_path, index=False)

print("Label-review audit saved to:")
print(review_path)
print("\nReview record:")
print(review_df.to_string(index=False))

print("\nOriginal test dataset remains unchanged.")


Label-review audit saved to:
/content/drive/MyDrive/Task1_Research/task1_test_label_review_audit.csv

Review record:
split                                text original_label suggested_label               review_status                                                                                                                                                                          reason  official_test_label_changed
 test what is the spanish word for garden       DATABASE         GENERAL Pending manual confirmation The query appears to request a translation. The documented labeling rules classify translations as GENERAL. Similar translation queries in the splits are also labeled GENERAL.                        False

Original test dataset remains unchanged.


In [21]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

results = [
    {
        "model": "TF-IDF + Logistic Regression",
        "test_accuracy_percent": 98.17,
        "correct": 1178,
        "test_samples": 1200
    },
    {
        "model": "Controlled raw-text Native SLM",
        "test_accuracy_percent": 99.67,
        "correct": 1196,
        "test_samples": 1200
    },
    {
        "model": "Original Native SLM",
        "test_accuracy_percent": 99.58,
        "correct": 1195,
        "test_samples": 1200
    },
    {
        "model": "Original KD Student",
        "test_accuracy_percent": 59.25,
        "correct": 711,
        "test_samples": 1200
    },
    {
        "model": "Fine-tuned 7B Teacher",
        "test_accuracy_percent": 99.83,
        "correct": 1198,
        "test_samples": 1200
    },
    {
        "model": "Fine-tuned Teacher + KD Student",
        "test_accuracy_percent": 99.92,
        "correct": 1199,
        "test_samples": 1200
    }
]

results_df = pd.DataFrame(results)

output_path = os.path.join(
    DATA_DIR,
    "task1_consolidated_results_audit.csv"
)

results_df.to_csv(output_path, index=False)

print("Consolidated results saved to:")
print(output_path)
print("\n", results_df.to_string(index=False))

print("\nImportant:")
print("- These are recorded experiment results, not a proof of generalization.")
print("- Verify each metric against its original prediction file.")
print("- The model input formats and evaluation procedures may differ.")
print("- The garden test label remains unchanged pending manual review.")


Consolidated results saved to:
/content/drive/MyDrive/Task1_Research/task1_consolidated_results_audit.csv

                           model  test_accuracy_percent  correct  test_samples
   TF-IDF + Logistic Regression                  98.17     1178          1200
 Controlled raw-text Native SLM                  99.67     1196          1200
            Original Native SLM                  99.58     1195          1200
            Original KD Student                  59.25      711          1200
          Fine-tuned 7B Teacher                  99.83     1198          1200
Fine-tuned Teacher + KD Student                  99.92     1199          1200

Important:
- These are recorded experiment results, not a proof of generalization.
- Verify each metric against its original prediction file.
- The model input formats and evaluation procedures may differ.
- The garden test label remains unchanged pending manual review.


In [22]:

import os
import pandas as pd
from sklearn.metrics import accuracy_score

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

prediction_files = {
    "Fine-tuned 7B Teacher":
        "finetuned_teacher_test_predictions_audit.csv",

    "Fine-tuned Teacher + KD Student":
        "finetuned_teacher_kd_student_test_predictions.csv",

    "Controlled raw-text Native SLM":
        "native_student_rawtext_controlled_test_predictions.csv",
}

for model_name, filename in prediction_files.items():
    path = os.path.join(DATA_DIR, filename)

    print(f"\n--- {model_name} ---")

    if not os.path.exists(path):
        print(f"Prediction file not found: {filename}")
        continue

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    # Detect the prediction column without assuming its name.
    pred_col = next(
        (c for c in [
            "predicted_label",
            "teacher_predicted_label",
            "prediction"
        ] if c in df.columns),
        None
    )

    if "label" not in df.columns or pred_col is None:
        print("Could not identify label/prediction columns.")
        continue

    accuracy = accuracy_score(df["label"], df[pred_col])

    print(f"Recomputed accuracy: {accuracy * 100:.4f}%")
    print("Prediction counts:")
    print(df[pred_col].value_counts(dropna=False))



--- Fine-tuned 7B Teacher ---
Rows: 1200
Columns: ['text', 'label', 'teacher_predicted_label', 'correct']
Recomputed accuracy: 99.8333%
Prediction counts:
teacher_predicted_label
GENERAL     600
DATABASE    600
Name: count, dtype: int64

--- Fine-tuned Teacher + KD Student ---
Rows: 1200
Columns: ['text', 'label', 'predicted_label', 'confidence']
Recomputed accuracy: 99.9167%
Prediction counts:
predicted_label
GENERAL     601
DATABASE    599
Name: count, dtype: int64

--- Controlled raw-text Native SLM ---
Prediction file not found: native_student_rawtext_controlled_test_predictions.csv


In [23]:

import os

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

print("Searching for Native SLM prediction files...\n")

matches = []

for root, dirs, files in os.walk(DATA_DIR):
    for filename in files:
        name = filename.lower()

        if (
            filename.lower().endswith(".csv")
            and any(word in name for word in ["native", "rawtext", "controlled"])
            and any(word in name for word in ["test", "prediction", "metric"])
        ):
            path = os.path.join(root, filename)
            matches.append(path)
            print(path)

if not matches:
    print("No matching CSV files found.")


Searching for Native SLM prediction files...

/content/drive/MyDrive/Task1_Research/native_student_test_metrics.csv


In [24]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

# 1. Read the existing Native SLM metrics
metrics_path = os.path.join(
    DATA_DIR, "native_student_test_metrics.csv"
)

print("=== Native SLM Test Metrics ===")
print(pd.read_csv(metrics_path).to_string(index=False))

# 2. List all files related to Native SLM
print("\n=== Native SLM Related Files ===")

for root, dirs, files in os.walk(DATA_DIR):
    for filename in files:
        if "native_student" in filename.lower():
            print(os.path.join(root, filename))


=== Native SLM Test Metrics ===
     model  accuracy  macro_precision  macro_recall  macro_f1
Native SLM  0.995833         0.995846      0.995833  0.995833

=== Native SLM Related Files ===
/content/drive/MyDrive/Task1_Research/native_student_training_history.csv
/content/drive/MyDrive/Task1_Research/native_student_validation_audit.csv
/content/drive/MyDrive/Task1_Research/native_student_test_metrics.csv


In [25]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

for filename in [
    "native_student_validation_audit.csv",
    "native_student_training_history.csv"
]:
    path = os.path.join(DATA_DIR, filename)

    print(f"\n===== {filename} =====")

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(5).to_string(index=False))



===== native_student_validation_audit.csv =====
Shape: (799, 6)
Columns: ['text', 'label', 'predicted_label', 'confidence_DATABASE', 'confidence_GENERAL', 'correct']
                                                                                                                              text    label predicted_label  confidence_DATABASE  confidence_GENERAL  correct
                                                                                                      what does idiosyncratic mean  GENERAL         GENERAL         1.328777e-06        9.999987e-01     True
                                                                                                      what is the weather in tokyo  GENERAL         GENERAL         8.315280e-07        9.999992e-01     True
What is the highest value for Byes, when Wins is less than 16, when Benalla DFL is "Swanpool", and when Against is less than 2177? DATABASE        DATABASE         1.000000e+00        1.621223e-08     True
         

In [26]:

import os
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATA_DIR = "/content/drive/MyDrive/Task1_Research"
path = os.path.join(DATA_DIR, "native_student_validation_audit.csv")

df = pd.read_csv(path)

print("Rows:", len(df))
print("Missing values:\n", df.isna().sum())
print("\nLabel counts:\n", df["label"].value_counts())
print("\nPrediction counts:\n", df["predicted_label"].value_counts())

# Recalculate correctness from the labels and predictions
recomputed_correct = (
    df["label"].astype(str).str.strip().str.upper()
    == df["predicted_label"].astype(str).str.strip().str.upper()
)

print("\nStored correct count:", df["correct"].sum())
print("Recomputed correct count:", recomputed_correct.sum())
print("Recomputed accuracy:", accuracy_score(
    df["label"], df["predicted_label"]
))

print("\nConfusion matrix [DATABASE, GENERAL]:")
print(confusion_matrix(
    df["label"],
    df["predicted_label"],
    labels=["DATABASE", "GENERAL"]
))

print("\nClassification report:")
print(classification_report(
    df["label"],
    df["predicted_label"],
    labels=["DATABASE", "GENERAL"],
    zero_division=0
))

print("\nRows marked incorrect:")
print(df.loc[~recomputed_correct, [
    "text", "label", "predicted_label", "correct"
]].to_string(index=False))


Rows: 799
Missing values:
 text                   0
label                  0
predicted_label        0
confidence_DATABASE    0
confidence_GENERAL     0
correct                0
dtype: int64

Label counts:
 label
DATABASE    400
GENERAL     399
Name: count, dtype: int64

Prediction counts:
 predicted_label
DATABASE    400
GENERAL     399
Name: count, dtype: int64

Stored correct count: 799
Recomputed correct count: 799
Recomputed accuracy: 1.0

Confusion matrix [DATABASE, GENERAL]:
[[400   0]
 [  0 399]]

Classification report:
              precision    recall  f1-score   support

    DATABASE       1.00      1.00      1.00       400
     GENERAL       1.00      1.00      1.00       399

    accuracy                           1.00       799
   macro avg       1.00      1.00      1.00       799
weighted avg       1.00      1.00      1.00       799


Rows marked incorrect:
Empty DataFrame
Columns: [text, label, predicted_label, correct]
Index: []


In [27]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

train = pd.read_csv(os.path.join(DATA_DIR, "task1_train.csv"))
val = pd.read_csv(os.path.join(DATA_DIR, "native_student_validation_audit.csv"))

def normalize_text(series):
    return (
        series.astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

train_text = set(normalize_text(train["text"]))
val_normalized = normalize_text(val["text"])

overlap_mask = val_normalized.isin(train_text)

print("Training rows:", len(train))
print("Validation rows:", len(val))
print("Exact normalized text overlaps:", int(overlap_mask.sum()))

if overlap_mask.any():
    print("\nOverlapping validation examples:")
    print(val.loc[overlap_mask, ["text", "label"]].head(20).to_string(index=False))
else:
    print("No exact normalized text overlap found.")


Training rows: 4000
Validation rows: 799
Exact normalized text overlaps: 0
No exact normalized text overlap found.


In [31]:


import os

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

print("Files containing 'native':")

for root, dirs, files in os.walk(DATA_DIR):
    dirs[:] = [
        d for d in dirs
        if not d.startswith("checkpoint-")
        and d not in ["node_modules", ".git"]
    ]

    for filename in files:
        if "native" in filename.lower():
            print(os.path.join(root, filename))


Files containing 'native':
/content/drive/MyDrive/Task1_Research/native_student_training_history.csv
/content/drive/MyDrive/Task1_Research/native_student_validation_audit.csv
/content/drive/MyDrive/Task1_Research/native_student_test_metrics.csv


In [32]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"
path = os.path.join(DATA_DIR, "native_student_test_metrics.csv")

df = pd.read_csv(path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.to_string(index=False))


Shape: (1, 5)
Columns: ['model', 'accuracy', 'macro_precision', 'macro_recall', 'macro_f1']
     model  accuracy  macro_precision  macro_recall  macro_f1
Native SLM  0.995833         0.995846      0.995833  0.995833


In [33]:

import os
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

files = {
    "Teacher": "finetuned_teacher_test_predictions_audit.csv",
    "KD Student": "finetuned_teacher_kd_student_test_predictions.csv"
}

for model_name, filename in files.items():
    path = os.path.join(DATA_DIR, filename)

    print(f"\n===== {model_name} =====")

    if not os.path.exists(path):
        print("Prediction file not found:", filename)
        continue

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    pred_col = (
        "teacher_predicted_label"
        if model_name == "Teacher"
        else "predicted_label"
    )

    if not {"text", "label", pred_col}.issubset(df.columns):
        print("Required columns missing.")
        continue

    accuracy = accuracy_score(df["label"], df[pred_col])
    print(f"Recomputed accuracy: {accuracy:.6%}")
    print("Missing values:\n", df[["text", "label", pred_col]].isna().sum())
    print("\nClassification report:")
    print(classification_report(
        df["label"], df[pred_col], zero_division=0
    ))

teacher = pd.read_csv(os.path.join(DATA_DIR, files["Teacher"]))
student = pd.read_csv(os.path.join(DATA_DIR, files["KD Student"]))

print("\n===== Cross-model alignment =====")
print("Same row count:", len(teacher) == len(student))
print("Same text in same order:",
      teacher["text"].equals(student["text"]))
print("Same labels in same order:",
      teacher["label"].equals(student["label"]))



===== Teacher =====
Rows: 1200
Columns: ['text', 'label', 'teacher_predicted_label', 'correct']
Recomputed accuracy: 99.833333%
Missing values:
 text                       0
label                      0
teacher_predicted_label    0
dtype: int64

Classification report:
              precision    recall  f1-score   support

    DATABASE       1.00      1.00      1.00       600
     GENERAL       1.00      1.00      1.00       600

    accuracy                           1.00      1200
   macro avg       1.00      1.00      1.00      1200
weighted avg       1.00      1.00      1.00      1200


===== KD Student =====
Rows: 1200
Columns: ['text', 'label', 'predicted_label', 'confidence']
Recomputed accuracy: 99.916667%
Missing values:
 text               0
label              0
predicted_label    0
dtype: int64

Classification report:
              precision    recall  f1-score   support

    DATABASE       1.00      1.00      1.00       600
     GENERAL       1.00      1.00      1.00       

In [34]:

import os
import json

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

folders_to_check = [
    "native_student_rawtext_controlled",
    "finetuned_teacher_kd_student",
    "finetuned_teacher_7b"
]

for folder in folders_to_check:
    path = os.path.join(DATA_DIR, folder)

    print("\n" + "=" * 65)
    print("FOLDER:", folder)
    print("=" * 65)

    if not os.path.exists(path):
        print("Folder not found:", path)
        continue

    for root, dirs, files in os.walk(path):
        dirs[:] = [
            d for d in dirs
            if not d.startswith("checkpoint-")
        ]

        for filename in files:
            full_path = os.path.join(root, filename)
            relative_path = os.path.relpath(full_path, path)

            print(relative_path)

            if filename in [
                "adapter_config.json",
                "config.json",
                "training_args.json"
            ]:
                try:
                    with open(full_path, "r") as f:
                        config = json.load(f)

                    print(json.dumps(config, indent=2)[:2500])
                except Exception as e:
                    print("Could not read config:", e)



FOLDER: native_student_rawtext_controlled
training_history.csv
test_predictions.csv
test_metrics.csv
checkpoints/README.md
checkpoints/adapter_model.safetensors
checkpoints/adapter_config.json
{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "Qwen/Qwen2.5-0.5B-Instruct",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "kasa_config": null,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 16,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "lora_ga_config": null,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "monteclora_config": null,
  "peft_type": "LORA",
  "peft_version": "0.21.1",
  "qalora_group_size": 16,
  "r": 8,
  "r

In [35]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

files_to_inspect = {
    "Native SLM test predictions":
        "native_student_rawtext_controlled/test_predictions.csv",
    "Native SLM test metrics":
        "native_student_rawtext_controlled/test_metrics.csv",
    "Native SLM training history":
        "native_student_rawtext_controlled/training_history.csv",
    "KD SLM test predictions":
        "finetuned_teacher_kd_student_test_predictions.csv",
    "KD SLM test metrics":
        "finetuned_teacher_kd_student_test_metrics.csv",
    "KD SLM training history":
        "finetuned_teacher_kd_student/training_history.csv"
}

for title, relative_path in files_to_inspect.items():
    path = os.path.join(DATA_DIR, relative_path)

    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

    if not os.path.exists(path):
        print("File not found:", path)
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))



Native SLM test predictions
Shape: (1200, 5)
Columns: ['text', 'label', 'actual_label', 'predicted_label', 'correct']
                                                                                                                    text    label actual_label predicted_label  correct
                                                                                   how many calories does an orange have  GENERAL      GENERAL         GENERAL     True
                                                                                                      let's hear a joke!  GENERAL      GENERAL         GENERAL     True
What is the sum of the pick numbers where the player was picked in round 5 and overall pick number was greater than 159? DATABASE     DATABASE        DATABASE     True

Native SLM test metrics
Shape: (1, 5)
Columns: ['model', 'test_accuracy', 'macro_precision', 'macro_recall', 'macro_f1']
                model  test_accuracy  macro_precision  macro_recall  macro_f1
Native SL

In [36]:

import os
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support
)

DATA_DIR = "/content/drive/MyDrive/Task1_Research"

native = pd.read_csv(os.path.join(
    DATA_DIR,
    "native_student_rawtext_controlled/test_predictions.csv"
))
kd = pd.read_csv(os.path.join(
    DATA_DIR,
    "finetuned_teacher_kd_student_test_predictions.csv"
))

# Check dataset alignment
print("Same number of rows:", len(native) == len(kd))
print("Same test texts in same order:",
      native["text"].equals(kd["text"]))
print("Same labels in same order:",
      native["label"].equals(kd["label"]))

# Check Native SLM's two label columns
print("\nNative label and actual_label agree:",
      (native["label"] == native["actual_label"]).all())

# Recalculate both models' metrics
for name, df, pred_col in [
    ("Native SLM", native, "predicted_label"),
    ("KD SLM", kd, "predicted_label")
]:
    y_true = df["label"]
    y_pred = df[pred_col]

    print(f"\n===== {name} =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Correct:", (y_true == y_pred).sum(), "/", len(df))

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred,
        labels=["DATABASE", "GENERAL"],
        zero_division=0
    )

    for i, label in enumerate(["DATABASE", "GENERAL"]):
        print(
            f"{label}: precision={precision[i]:.6f}, "
            f"recall={recall[i]:.6f}, F1={f1[i]:.6f}"
        )

    print("Confusion matrix [DATABASE, GENERAL]:")
    print(confusion_matrix(
        y_true, y_pred,
        labels=["DATABASE", "GENERAL"]
    ))

# Compare error cases
native_correct = native["label"] == native["predicted_label"]
kd_correct = kd["label"] == kd["predicted_label"]

print("\n===== Error comparison =====")
print("Native wrong, KD correct:",
      ((~native_correct) & kd_correct).sum())
print("Native correct, KD wrong:",
      (native_correct & (~kd_correct)).sum())
print("Both wrong:",
      ((~native_correct) & (~kd_correct)).sum())


Same number of rows: True
Same test texts in same order: True
Same labels in same order: True

Native label and actual_label agree: True

===== Native SLM =====
Accuracy: 0.9966666666666667
Correct: 1196 / 1200
DATABASE: precision=0.995017, recall=0.998333, F1=0.996672
GENERAL: precision=0.998328, recall=0.995000, F1=0.996661
Confusion matrix [DATABASE, GENERAL]:
[[599   1]
 [  3 597]]

===== KD SLM =====
Accuracy: 0.9991666666666666
Correct: 1199 / 1200
DATABASE: precision=1.000000, recall=0.998333, F1=0.999166
GENERAL: precision=0.998336, recall=1.000000, F1=0.999167
Confusion matrix [DATABASE, GENERAL]:
[[599   1]
 [  0 600]]

===== Error comparison =====
Native wrong, KD correct: 3
Native correct, KD wrong: 0
Both wrong: 1
